# 텍스트 마이닝을 활용한 소비패턴 분석 서비스
- 사용자에게 상세 지출 내역을 작성받고, 사용자가 작성한 텍스트를 분석
- 충동구매 대표 문장을 통해 유사도 파악 + 충동구매 키워드 추출 -> 텍스트 분석
- 이용 데이터 : navershopping.txt

=> textAnalyzer의 가이드라인(함수)을 이용해서 Flask에서 유사도 분석 처리!

### 1. 준비 단계
- 라이브러리 불러오기 및 분석기 미리 꺼내두기

In [1]:
# 필요한 라이브러리 불러오기
import os # 시스템 환경변수 관리
import pandas as pd # 데이터프레임
from konlpy.tag import Okt # 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer # 벡터화
from sklearn.linear_model import LogisticRegression # 충동구매 여부를 0과 1로 분류할 모델
from sklearn.model_selection import train_test_split # 전체 데이터를 학습용과 테스트 용으로 분리
from tqdm import tqdm # 로딩바
import joblib # 모델 저장 및 불러오기용 (.pkl)

# 자바 경로 환경 설정
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-21'

# 형태소 분석기 및 로딩바 연결하기
okt = Okt()
tqdm.pandas() # tqdm은 pd를 모름! pandas라고 써줘야함.

### 2. 데이터 전처리 및 학습
- 데이터 읽어와서 모델 학습
- 가장 오래 걸림!

In [2]:
# 데이터 불러오기
df = pd.read_table('./data/naver_shopping.txt', 
                     sep= '\t', 
                     names = ['rating', 'review'], 
                     encoding = 'utf-8')

# 충동구매 의심 키워드
impulse_keywords = ['충동', '지름신', '홧김', '나도모르게', '무계획',
                    '지르다', '홀리다', '세일', '우울', '그냥', '과소비',
                    '싶다', '짜증', '스트레스', '모르다', '혹해서',
                    '플렉스', '땡기다', '나를 위한', '긁다', '현타',
                    '허하다', '유튜버', '유행', '순간', '보자마자', '꿀꿀',
                    '싱숭생숭', '멜랑콜리', '억울', '갑갑', '답답', '슬프다', 
                    '힘들다', '울적']

# 보상심리 전용 키워드
reward_keywords = ['나를 위한', '보상', '고생', '수고']


# 특정 조합 규칙 함수
def logic(text):
    # 정규화 및 어간 추출 (걍 -> 그냥, 샀다 -> 사다)
    # stem = True -> 기본형으로 통일시키겠다!
    morphs = okt.morphs(okt.normalize(str(text)), stem=True)

    # 1. '그냥' + 구매 => 충동구매
    if '그냥' in morphs and any(buy in morphs for buy in ['사다', '구매', '구입', '결제', '지르다', '주문', '먹다']):
        return 'impulse'

    # 2. '선물' + '나' => 보상심리
    if '선물' in morphs and '나' in morphs:
        return 'reward'

    # 3. 감정 키워드 + 구매 => 충동구매
    emotion_bad = ['우울', '울적', '스트레스', '짜증', '홧김', '답답',
                   '갑갑', '싱숭생숭', '멜랑콜리', '억울', '꿀꿀', 
                   '슬프다', '힘들다']
    purchase_action = ['사다', '구매', '구입', '결제', '지르다', '주문', '먹다']

    if any(emo in morphs for emo in emotion_bad) and any(buy in morphs for buy in purchase_action):
        return 'impulse'

    return None # 조합에 걸리지 않으면 None 반환
    

# 라벨링 함수
def labeling (row):
    text = str(row['review'])

    # 1. 조합 로직 먼저 체크 (그냥 + 샀다, 나를 위한 + 선물)
    logic_res = logic(text)

    is_reward = 0
    has_impulse = 0
    
    if logic_res == 'reward':
        is_reward = 1
    elif logic_res == 'impulse':
        has_impulse = 1

    # 2. 조합에 걸리지 않았을 때 => 단일 키워드 체크
    if not is_reward and not has_impulse:
        # 보상심리 체크
        is_reward = 1 if any (word in text for word in reward_keywords) else 0
        # 충동구매 체크
        has_impulse = 1 if any (word in text for word in impulse_keywords) else 0

    # 3. 최종 학습 라벨 결정
    # 감정 지출 (보상 or 충동) => label = 1 혹은 평점 1~2점일 때 1
    final_label = 1 if (is_reward or has_impulse or row ['rating'] <= 2) else 0

    # 반환값 : label(y) 값과 DB 통계용 (is_reward) 두 값 반환
    return pd.Series([final_label, is_reward])


# 적용 및 전처리 실행
df[['label', 'is_reward']] = df.progress_apply(labeling, axis=1)


# 형태소 분석
def tokenize(text):
    clean_text = okt.normalize(str(text))
    return ' '.join(okt.morphs(clean_text, stem=True))

df['tokenized_review'] = df['review'].progress_apply(tokenize)

100%|██████████| 200000/200000 [07:49<00:00, 425.85it/s]


In [3]:
print(df[['review', 'tokenized_review']].head())

                                              review  \
0                                            배공빠르고 굿   
1                      택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고   
2  아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...   
3  선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...   
4                  민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ   

                                    tokenized_review  
0                                           배공 빠르다 굿  
1               택배 가 엉망 이네 용 저희 집 밑 에 층 에 말 도 없이 놔두다  
2  아주 좋다 바지 정말 좋다 2 개 더 구매 하다 이 가격 에 대박 이다 . 바느질 ...  
3  선물 용 으로 빨리 받다 전달 하다 하다 상품 이다 머그컵 만 오다 당황 하다 . ...  
4            민트 색상 예쁘다 . 옆 손잡이 는 거 는 용 도로 도 사용 되다 ㅎㅎ  


In [24]:
# 모델 학습

# 불용어 리스트
custom_stopwords = [
    '좋다', '맛있다', '최고', '감사하다', '추천', '괜찮다', '만족', '역시', '자주',
    '배송', '빠르다', '도착', '포장', '택배', '상태', '문의', '하다', '있다', '되다', '같다',
    '이다', '의', '가', '이', '은', '는', '을', '를', '에', '도', '로', '으로',
    '것', '수', '거', '나', '저', '분', '때', '점', '요', '서', '게', '진짜', '정말', '너무'
]

# 단어 벡터화 (TF-IDF)
vectorizer = TfidfVectorizer(max_features=20000,
                             ngram_range=(1, 3), 
                             stop_words=custom_stopwords)
X = vectorizer.fit_transform(df['tokenized_review'])
y = df['label']


# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 2026)

# 모델 학습
# class_weight = 'balanced' -> 적은 양의 충동구매 데이터를 더 엄격하게 학습시킴
# C = 1.0 -> 규제 강도 (기본값 1.0에서 조절 가능, 높게 설정하면 더 세밀하게 학습함!)
model = LogisticRegression(max_iter = 2000,
                           class_weight = 'balanced', 
                           C = 50.0,
                           solver = 'liblinear')
model.fit(X_train, y_train)

# 빈틈 교정 함수 (
def predict_smart(text):
    clean_text = ' '.join(okt.morphs(okt.normalize(str(text)), stem=True))
    
    # 강력 교정 키워드
    correction_keywords = ['우울', '울적', '스트레스', '홧김', '꿀꿀', '슬프다', '힘들다', '답답']
    
    if any(word in text for word in correction_keywords) or any(word in clean_text for word in correction_keywords):
        return 1, 100.0
    
    vec = vectorizer.transform([clean_text])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    return int(pred), float(max(prob) * 100)

# 정확도 확인
score = model.score(X_test, y_test)
accuracy = score * 100
print("정확도: ", accuracy)

정확도:  86.7


In [25]:
# [실전 테스트 함수]
def predict_meokbap(text):
    # 1. 입력받은 문장을 AI가 배운 방식(TF-IDF)으로 벡터화
    # 리스트 형태로 감싸서 넣어줘야 합니다! (자바의 Array처럼요)
    vec = vectorizer.transform([text]) 
    
    # 2. 모델 예측 (0: 일반지출, 1: 충동/보상지출)
    pred = model.predict(vec)[0]
    
    # 3. 확률 확인 (AI가 얼마나 확신하는지 - 자바의 정확도 체크 느낌!)
    prob = model.predict_proba(vec)[0]
    confidence = max(prob) * 100
    
    result = "🔥 충동/보상 지출" if pred == 1 else "✅ 일반 지출"
    print(f"입력: {text}")
    print(f"결과: {result} (확신도: {confidence:.2f}%)")
    print("-" * 30)

# --- 유저님이 테스트하고 싶은 문장들을 아래에 넣어보세요! ---
test_list = [
    "스트레스 받아서 걍 편의점 털었음",
    "나를 위한 선물로 에어팟 프로 샀다",
    "오늘 점심은 김치찌개 먹었는데 맛있네", # 이건 0이 나와야 정상!
    "홧김에 가방 질렀는데 후회 중",
    "이번 달 고생한 나에게 주는 보상",
    "우울해서 빵 사먹음",
]

for t in test_list:
    predict_meokbap(t)

입력: 스트레스 받아서 걍 편의점 털었음
결과: 🔥 충동/보상 지출 (확신도: 99.99%)
------------------------------
입력: 나를 위한 선물로 에어팟 프로 샀다
결과: 🔥 충동/보상 지출 (확신도: 66.90%)
------------------------------
입력: 오늘 점심은 김치찌개 먹었는데 맛있네
결과: ✅ 일반 지출 (확신도: 83.89%)
------------------------------
입력: 홧김에 가방 질렀는데 후회 중
결과: ✅ 일반 지출 (확신도: 51.58%)
------------------------------
입력: 이번 달 고생한 나에게 주는 보상
결과: 🔥 충동/보상 지출 (확신도: 99.95%)
------------------------------
입력: 우울해서 빵 사먹음
결과: ✅ 일반 지출 (확신도: 64.98%)
------------------------------


In [19]:
# logic 함수에서 '우울'이 포함된 리뷰가 정말 label 1로 잘 갔는지 확인!
print(df[df['review'].str.contains("우울")].head())

       rating                                             review  label  \
1470        5  너무좋아서 어쪄죠 맘이 떨어 지지지지 너무 좋아여 제품 상태가 좋네요 폴대가 좀 찌...      1   
8859        5            라넌큘러스가 집안분위기는 물론 우울했던 기분까지 환하게 바꿔줬어요~~♥      1   
11887       5  즐거워용. 우울함이 오래 있었거든요.뭐든 해야지 했는데 우연히 방송에서 봤어요.독학...      1   
25563       2  뚱뚱한 사람은 우울... 뱃살이 불룩불룩 하네요ㅠ 허리 부분이 넓은 밴드형태면 깔끔...      1   
28276       2  뒤꿈치개아파여..,.다까져서피철철ㅜㅜ하 내발이문제인가이게문제인건가ㅜㅜ딴건안그랫는데ㅜ...      1   

       is_reward                                   tokenized_review  
1470           0  너무 좋다 어쪄 죠 맘 이 떨다 지지 지 너무 좋다 제품 상태 가 좋다 폴 대가 좀...  
8859           0     라넌큘러스 가 집안 분위기 는 물론 우울하다 기분 까지 환하다 바꾸다 주다 ~~ ♥  
11887          0  즐겁다 . 우울함 이 오래 있다 . 뭐 든 하다 하다 우연히 방송 에서 보다 . 독...  
25563          0  뚱뚱하다 사람 은 우울 ... 뱃살 이 불 룩 불 룩 하다 ㅠ 허리 부분 이 넓다 ...  
28276          0  뒤꿈치 개 아 파이다 ..,. 다 끄다 피철철 ㅜㅜ 하 내 발 이 문제 인가 이 게...  


### 3. pkl 파일로 저장
- 모델 학습 단계를 없애서 시간 단축시키기!

In [26]:
# 모델 저장
joblib.dump(model, 'analyzer_meokbap.pkl')
joblib.dump(vectorizer, 'tfidf.pkl')

['tfidf.pkl']